# Using Remote Evaluators (Managed LLM-as-Judge)

In this notebook, we'll walk through how to use `RemoteEvaluator` to run **LLM-as-Judge evaluations configured in the Datadog UI** directly from your local experiments — without reimplementing the evaluation logic in Python.

### What is RemoteEvaluator?

`RemoteEvaluator` lets you reference a [custom LLM-as-a-judge evaluation](https://docs.datadoghq.com/llm_observability/evaluations/custom_llm_as_a_judge_evaluations) by name, and run it as part of a local experiment. This means you can:

- **Reuse production evaluators** in offline experiments without copy-pasting prompt logic
- **Keep evaluation logic centralized** in the Datadog UI where it can be versioned and shared
- **Mix remote and local evaluators** in the same experiment for comprehensive testing

### Prerequisites

For sections 2–4, you need a custom LLM-as-a-judge evaluation already configured in the [Datadog UI](https://app.datadoghq.com/llm/evaluations):
1. Create the evaluator in the UI and take note of its name.
2. Ensure it is **enabled** and has a properly configured prompt template.

> **No evaluator in the UI yet?** Section 5 shows how to define one in code and publish it to the UI using `LLMObs.publish_evaluator()`.

### Learning Goals

- Understand how to use `RemoteEvaluator` in experiments
- Learn how to customize input mapping with `transform_fn`
- Mix remote and local evaluators in the same experiment
- Publish a locally-defined `LLMJudge` evaluator to the Datadog UI with `LLMObs.publish_evaluator()`

## 0. Setup

First, let's initialize the required libraries.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

from typing import Any

from ddtrace.llmobs import EvaluatorResult, LLMObs, RemoteEvaluator
from openai import OpenAI

LLMObs.enable(
    api_key=os.getenv("DD_API_KEY"),
    app_key=os.getenv("DD_APPLICATION_KEY"),
    project_name="Onboarding",
    ml_app="Onboarding-ML-App",
)

oai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. Create a Dataset

We'll create a small dataset of geography questions to use with our evaluators. Each record has an `input_data` with a question, and an `expected_output` with the correct answer.

In [ ]:
dataset = LLMObs.create_dataset(
    dataset_name="capitals-of-the-world",
    description="a list of inputs and outputs describing capitals of the world",
    records=[
        {
            "input_data": {"question": "What is the capital of China?"},
            "expected_output": "Beijing",
            "metadata": {"difficulty": "easy"},
        },
        {
            "input_data": {
                "question": "Which city serves as the capital of South Africa?"
            },
            "expected_output": "Pretoria",
            "metadata": {"difficulty": "medium"},
        },
    ],
)

dataset.as_dataframe()

We also need a task function. We'll reuse a simple capital-generating task.

In [ ]:
def generate_capital(input_data: dict[str, Any], config: dict[str, Any]) -> str:
    output = oai_client.chat.completions.create(
        model=config["model"],
        messages=[{"role": "user", "content": input_data["question"]}],
    )
    return output.choices[0].message.content

## 2. Basic Usage

The simplest way to use `RemoteEvaluator` is to reference an evaluator by name. This name must match the name of a custom LLM-as-a-judge evaluation that you've already configured in the Datadog UI.

By default, `RemoteEvaluator` automatically maps:
- `input_data` from your dataset record &rarr; `{{span_input}}` in the prompt template
- `output_data` (your task's return value) &rarr; `{{span_output}}` in the prompt template
- `expected_output` from your dataset record &rarr; `{{meta.expected_output}}` in the prompt template

> **Note**: Replace `"your-evaluator-name"` below with the actual name of an evaluator you've created in the Datadog UI.

In [ ]:
# Replace with the name of your LLM-as-Judge evaluator from the Datadog UI
EVALUATOR_NAME = "your-evaluator-name"

remote_eval = RemoteEvaluator(eval_name=EVALUATOR_NAME)

experiment = LLMObs.experiment(
    name="remote-eval-basic",
    dataset=dataset,
    task=generate_capital,
    evaluators=[remote_eval],
    config={"model": "gpt-5-mini"},
)

results = experiment.run(jobs=1)
experiment.url

After a few seconds, you should see the experiment in Datadog with evaluation results from your remote evaluator. The evaluation scores, reasoning, and assessments are all generated by the LLM-as-Judge configuration you set up in the UI.

## 3. Custom Input Mapping with `transform_fn`

When you configure an LLM-as-Judge in the Datadog UI, the [prompt template uses variables](https://docs.datadoghq.com/llm_observability/evaluations/custom_llm_as_a_judge_evaluations#configure-the-prompt) such as `{{span_input}}` and `{{span_output}}`.

By default, `RemoteEvaluator` maps:
- `input_data` &rarr; `span_input`
- `output_data` &rarr; `span_output`

But if your dataset records have a different structure — for example, `input_data` is a dict with multiple keys — you can provide a `transform_fn` to control exactly which values are sent for each template variable.

The `transform_fn` receives an `EvaluatorContext` and returns a dict whose keys map to the prompt template variables.

In [ ]:
from ddtrace.llmobs import EvaluatorContext


def my_transform(context: EvaluatorContext) -> dict:
    """Map dataset fields to the prompt template variables.

    For a prompt template that uses:
      - {{span_input}} for the user's question
      - {{span_output}} for the model's answer
      - {{meta.expected_output}} for the ground truth
    """
    return {
        "span_input": context.input_data.get(
            "question"
        ),  # extract just the question string
        "span_output": context.output_data,  # the task's return value
        "meta": {
            "expected_output": context.expected_output,  # ground truth from dataset
        },
    }


remote_eval_with_transform = RemoteEvaluator(
    eval_name=EVALUATOR_NAME,
    transform_fn=my_transform,
)

experiment = LLMObs.experiment(
    name="remote-eval-with-transform",
    dataset=dataset,
    task=generate_capital,
    evaluators=[remote_eval_with_transform],
    config={"model": "gpt-5-mini"},
)

results = experiment.run(jobs=1)
experiment.url

The `transform_fn` is especially useful when:
- Your `input_data` is a dict with multiple keys and you only want to send specific fields
- Your prompt template uses custom variables beyond `span_input` and `span_output` (e.g., `{{meta.retrieved_docs}}`)
- You need to pre-process or format data before sending it to the evaluator

## 4. Mixing Remote and Local Evaluators

One of the most powerful aspects of `RemoteEvaluator` is that you can combine it with local evaluators in the same experiment. This lets you run your centralized LLM-as-Judge evaluation alongside quick local checks.

In [ ]:
# A simple local evaluator: checks if the expected answer appears in the output
def contains_answer(input_data, output_data, expected_output):
    found = expected_output in output_data
    return EvaluatorResult(
        value=found,
        reasoning=f"Expected '{expected_output}' {'found' if found else 'not found'} in output",
        assessment="pass" if found else "fail",
    )


# A simple local evaluator: exact string match
def exact_match(input_data, output_data, expected_output):
    return expected_output == output_data


# Combine remote + local evaluators
remote_eval = RemoteEvaluator(eval_name=EVALUATOR_NAME)

experiment = LLMObs.experiment(
    name="remote-and-local-evaluators",
    dataset=dataset,
    task=generate_capital,
    evaluators=[remote_eval, contains_answer, exact_match],
    config={"model": "gpt-5-mini"},
)

results = experiment.run(jobs=5)
experiment.url

## 5. Publishing a Local Evaluator to the UI

So far we've been using evaluators that were already configured in the Datadog UI. But you can also go the other direction: **define an evaluator locally in code and publish it to the UI** using `LLMObs.publish_evaluator()`.

This is useful when you want to:
- **Version-control your evaluator logic** alongside your codebase
- **Bootstrap a new evaluator** from an experiment, then reuse it in production monitoring
- **Share evaluators** with teammates by publishing them once and referencing by name

The flow is:
1. Define an `LLMJudge` evaluator locally (with prompt, model, and structured output)
2. Call `LLMObs.publish_evaluator()` to register it in the Datadog UI
3. Immediately use it as a `RemoteEvaluator` by name

> **Note**: The published evaluator starts **disabled** by default. You need to enable it in the [Datadog UI](https://app.datadoghq.com/llm/evaluations) before `RemoteEvaluator` can run it against live spans. For use in experiments (as shown below), it runs locally so the enabled/disabled state doesn't matter.

In [ ]:
from ddtrace.llmobs import BooleanStructuredOutput, LLMJudge

# 1. Define the evaluator locally in code
judge = LLMJudge(
    name="capital_correctness",
    system_prompt="You are a geography expert. Evaluate whether the model's answer is correct.",
    user_prompt=(
        "The user asked: {{input_data}}\n"
        "The model answered: {{output_data}}\n"
        "The expected answer is: {{expected_output}}\n\n"
        "Is the model's answer correct?"
    ),
    structured_output=BooleanStructuredOutput(
        description="True if the answer correctly identifies the capital city, False otherwise.",
        reasoning=True,
        pass_when=True,
    ),
    provider="openai",
    model="gpt-4o-mini",
)

# 2. Publish it to the Datadog UI
result = LLMObs.publish_evaluator(
    evaluator=judge,
    ml_app="Onboarding-ML-App",
)

print("Evaluator published! View it at:", result["ui_url"])

# 3. Use it immediately in an experiment as a RemoteEvaluator
#    (Enable it in the UI first if you want it to run against live production spans)
published_remote_eval = RemoteEvaluator(eval_name="capital_correctness")

experiment = LLMObs.experiment(
    name="publish-and-run-evaluator",
    dataset=dataset,
    task=generate_capital,
    evaluators=[published_remote_eval],
    config={"model": "gpt-4o-mini"},
)

results = experiment.run(jobs=1)
experiment.url

## Summary

In this notebook, we covered:

| Concept | Description |
|---------|-------------|
| **Basic usage** | Reference an evaluator by name with `RemoteEvaluator(eval_name="...")` |
| **`transform_fn`** | Customize how dataset fields map to prompt template variables |
| **Mixed evaluators** | Combine remote and local evaluators in the same experiment |
| **`publish_evaluator`** | Define an `LLMJudge` locally and publish it to the Datadog UI for reuse |

### Key takeaways

- `RemoteEvaluator` bridges the gap between evaluations configured in the UI and local experiment workflows
- You don't need to reimplement prompt logic — just reference the evaluator by name
- `LLMObs.publish_evaluator()` lets you go the other direction: define evaluator logic in code and register it in the UI

---

For more details, see:
- [Custom LLM-as-a-Judge Evaluations](https://docs.datadoghq.com/llm_observability/evaluations/custom_llm_as_a_judge_evaluations)
- [Evaluation Developer Guide — Using Managed Evaluators](https://docs.datadoghq.com/llm_observability/guide/evaluation_developer_guide/#using-managed-evaluators)
- [LLM Experiments](https://docs.datadoghq.com/llm_observability/experiments)